# Multiple Time Intervals

You may want to make trading decisions on a different timeframe than your underlying data source. For instance, you might choose to execute trades on daily bars after confirming the trend on weekly or monthly bars. **PyBroker v2** supports compressing backtest data into coarser intervals and making those compressed bars available to your strategy.

## Interval Types

You can define an interval using any of these three formats:

*   **Every-n-bars** (`int` greater than `1`): Compresses every `n` base bars into one bar. For example, using `5` on daily data produces one bar per five trading days.
*   **Duration** (`str`): A fixed time span written as digits followed by a single unit letter (`s`, `m`, `h`, or `d`). For example, `"5m"` compresses 1-minute bars into 5-minute bars.
*   **Calendar** (`str`): Aligns exactly to standard calendar boundaries using one of the following options:

| Calendar String | Boundary Alignment |
| :--- | :--- |
| `"daily"` | Standard daily boundary. |
| `"weekly"` | Starts on Monday. |
| `"monthly"` | Starts on the 1st of the month. |
| `"quarterly"` | Begins in January, April, July, and October. |
| `"yearly"` | Starts on January 1. |

Your chosen interval must always be strictly coarser than the bars being compressed. For example, if you fetch daily bars from [YFinance](https://www.pybroker.com/en/latest/reference/pybroker.data.html#pybroker.data.YFinance), `"weekly"` and `"monthly"` are valid intervals. Attempting to use `"daily"` or `"1h"` will raise a `ValueError`.

Before using intervals in a strategy, let's build some intuition by compressing bars directly. We will start by downloading daily data:

In [1]:
import pybroker
from pybroker import Strategy, YFinance

pybroker.enable_data_source_cache("multiple_time_intervals")

yfinance = YFinance()
df = yfinance.query(
    ["AMD", "NVDA", "INTC"], start_date="1/1/2021", end_date="1/1/2026"
)
df.head()

Loading bar data...


[                       0%                       ]

[**********************67%*******                ]  2 of 3 completed

[*********************100%***********************]  3 of 3 completed

Loaded bar data: 0:00:00 



,date,symbol,open,high,low,close,volume,adj_close
0,2021-01-04,AMD,92.110001,96.059998,90.919998,92.300003,51802600,92.300003
1,2021-01-04,INTC,49.889999,51.389999,49.400002,49.669998,46102500,44.902935
2,2021-01-04,NVDA,13.104250,13.652500,12.962500,13.113500,560640000,13.060798
3,2021-01-05,AMD,92.099998,93.209999,91.410004,92.769997,34208000,92.769997
4,2021-01-05,INTC,49.450001,50.830002,49.330002,50.610001,24866600,45.752724


## Compressing Bars

The [compress_bars](https://www.pybroker.com/en/latest/reference/pybroker.interval.html#pybroker.interval.compress_bars) function converts single-symbol OHLCV data (either a [Pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) or [BarData](https://www.pybroker.com/en/latest/reference/pybroker.common.html#pybroker.common.BarData)) to a coarser interval, returning the result as a new [BarData](https://www.pybroker.com/en/latest/reference/pybroker.common.html#pybroker.common.BarData) object. Every compressed bar is timestamped with the date of the last base bar it contains.

When grouping base bars into a compressed bar, the data is aggregated as follows:
*   **Open:** Taken from the first base bar.
*   **High / Low:** The highest high and lowest low.
*   **Close:** Taken from the last base bar.
*   **Volume:** The sum of the volumes.
*   **VWAP:** The volume-weighted average.
*   **Custom columns:** The last value in the period (e.g., [YFinance](https://www.pybroker.com/en/latest/reference/pybroker.data.html#pybroker.data.YFinance)'s `adj_close`).

You must also supply the `base_timeframe` parameter to declare the spacing of your input bars (for example, `"1d"` for daily data). 

Let's compress AMD into calendar weeks and view the result as a [Pandas DataFrame](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.html) with [bars_to_df](https://www.pybroker.com/en/latest/reference/pybroker.common.html#pybroker.common.bars_to_df):

In [2]:
from pybroker import compress_bars
from pybroker.common import bars_to_df


amd_df = df[df["symbol"] == "AMD"]
bars_to_df(compress_bars(amd_df, "weekly", base_timeframe="1d")).head()

,date,open,high,low,close,volume,adj_close
0,2021-01-08,92.110001,96.400002,89.459999,94.580002,220635900.0,94.580002
1,2021-01-15,94.029999,99.230003,87.860001,88.209999,279733900.0,88.209999
2,2021-01-22,89.559998,95.949997,87.239998,92.790001,205817500.0,92.790001
3,2021-01-29,94.139999,95.739998,85.019997,85.639999,291661400.0,85.639999
4,2021-02-05,86.830002,89.480003,84.660004,87.900002,169582500.0,87.900002


Every-n-bars compression works the same way. In this example, every `5` daily bars become one bar:

In [3]:
bars_to_df(compress_bars(amd_df, 5, base_timeframe="1d")).head()

,date,open,high,low,close,volume,adj_close
0,2021-01-08,92.110001,96.400002,89.459999,94.580002,220635900.0,94.580002
1,2021-01-15,94.029999,99.230003,87.860001,88.209999,279733900.0,88.209999
2,2021-01-25,89.559998,95.949997,87.239998,94.129997,260904400.0,94.129997
3,2021-02-01,94.910004,95.720001,84.660004,87.660004,278933800.0,87.660004
4,2021-02-08,88.489998,91.989998,86.879997,91.470001,174863100.0,91.470001


## A Multi-Timeframe Strategy

When building a backtest, you can declare higher timeframes by passing the `intervals` parameter to [add_execution](https://www.pybroker.com/en/latest/reference/pybroker.strategy.html#pybroker.strategy.Strategy.add_execution). Your execution function can then read these compressed bars through [ctx.interval](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.interval), which returns a read-only [IntervalContext](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.IntervalContext).

Both indicators and models attached to the execution are automatically computed on the base timeframe and across every declared interval. For example, an indicator with a `period=10` will instantly have a 10-week version available on the weekly bars, made accessible via the [indicator](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.IntervalContext.indicator) method on that context (`ctx.interval("weekly").indicator(...)`).

To prevent look-ahead bias, [ctx.interval](https://www.pybroker.com/en/latest/reference/pybroker.context.html#pybroker.context.ExecContext.interval) only ever exposes *completed* bars. The week or month that is currently forming is never visible, ensuring that future data cannot leak into your daily trading decisions.

To demonstrate this, we will execute trades on daily bars and assign a specific job to each higher timeframe:

*   **Monthly (Regime):** Only enter when the last completed monthly close is higher than the close from three months ago.
*   **Weekly (Trend):** Only enter when the last completed weekly close is above its 10-week moving average, and exit when it falls below that average.
*   **Daily (Timing):** Enter on the first daily close that crosses above the weekly moving average:

In [4]:
from pybroker.vect import sumv


sma_10 = pybroker.indicator("sma_10", lambda data: sumv(data.close, 10) / 10)


def buy_with_trend(ctx):
    weekly = ctx.interval("weekly")
    monthly = ctx.interval("monthly")
    # Wait until enough completed weekly and monthly bars exist.
    if len(weekly.close) == 0 or len(monthly.close) < 4:
        return
    wk_sma = weekly.indicator("sma_10")
    regime_up = monthly.close[-1] > monthly.close[-4]
    trend_up = weekly.close[-1] > wk_sma[-1]
    pos = ctx.long_pos()
    if not pos and regime_up and trend_up and ctx.close[-1] > wk_sma[-1]:
        ctx.buy_shares = 100
    elif pos and not trend_up:
        ctx.sell_all_shares()


strategy = Strategy(yfinance, start_date="1/1/2021", end_date="1/1/2026")
strategy.add_execution(
    buy_with_trend,
    ["AMD", "NVDA", "INTC"],
    indicators=sma_10,
    intervals=["weekly", "monthly"],
)
result = strategy.backtest(timeframe="1d")
result.metrics_df.head(20)

Backtesting: 2021-01-01 00:00:00 to 2026-01-01 00:00:00



Loaded cached bar data.



Computing indicators...


  0% (0 of 9) |                          | Elapsed Time: 0:00:00 ETA:  --:--:--

 11% (1 of 9) |##                        | Elapsed Time: 0:00:00 ETA:   0:00:02

...

  0% (0 of 1255) |                       | Elapsed Time: 0:00:00 ETA:  --:--:--

 67% (851 of 1255) |##############       | Elapsed Time: 0:00:00 ETA:   0:00:00

100% (1255 of 1255) |####################| Elapsed Time: 0:00:00 Time:  0:00:00

Finished backtest: 0:00:01


,name,value
0,trade_count,36
1,initial_market_value,100000.0
2,end_market_value,120278.0
3,total_pnl,20361.0
4,unrealized_pnl,-83.0
5,total_return_pct,20.361
6,total_profit,31960.0
7,total_loss,-11599.0
8,total_fees,0.0
9,max_drawdown,-9555.0
